# B3 콘텐츠 기반 추천기 확인

B1 태그 정제, B2 책 벡터와 공통 Recommender 인터페이스·인기순 기준선·평가 함수를 사용합니다.
첫 설정의 연결과 성능을 확인하는 검증 실험이며, 최종 설정 선택이나 최종 평가가 아닙니다.
기존 조원 파일과 원본 데이터는 수정하지 않습니다.


In [1]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from bookrec.data import load_book_tags, load_books, load_split, load_to_read
from bookrec.data.tag_filter import filter_book_tags
from bookrec.models.content_based.features import build_book_features
from bookrec.models.content_based.recommender import ContentBasedRecommender
from bookrec.models.popularity import PopularityRecommender
from bookrec.evaluation import evaluate_warm, evaluate_cold

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
split = load_split()
to_read = load_to_read()
features = build_book_features(filter_book_tags(load_book_tags()))
print(f"학습 평점 {len(split.train):,}건, 특징 행렬 {features.matrix.shape}")


학습 평점 3,769,078건, 특징 행렬 (10000, 16958)


## 1. 모델 준비

positive 가중치는 `max(rating - 3, 0)`입니다. 4점 책은 1, 5점 책은 2로 가중합하고,
독자 벡터를 L2 정규화한 뒤 책 벡터와 내적합니다. 모든 점수는 0~1입니다.
독자가 좋아한 책이 없거나 그 책의 특징이 비면 train 평점 개수 / 최대 평점 개수로 대체합니다.
이력이 1개뿐인 독자도 처리하며, 1~3점 이력만 있으면 개인 취향을 추정하지 않습니다.


In [2]:
models = {
    "popularity": PopularityRecommender(),
    "cb": ContentBasedRecommender(features, profile_weighting="positive"),
}
for name, model in models.items():
    start = perf_counter()
    model.fit(split.train)
    print(f"{name}: fit {perf_counter() - start:.3f}초")


popularity: fit 0.008초
cb: fit 0.017초


## 2. 첫 검증 평가

기존 독자는 `val`에서 seed=42인 동일한 1,000명, 신규 독자는 `cold_tune` 전원(2,500명)을 사용합니다.
각 모델은 train으로만 학습합니다. 신규 독자의 1·3·5·10개 이력은 같은 정답(11번째 이후의 4점 이상 책)으로 평가합니다.
test와 cold_final 평가 함수는 호출하지 않습니다. 두 목표의 점수 크기를 직접 비교하지 않습니다.


In [3]:
records = []
for name, model in models.items():
    start = perf_counter()
    warm = evaluate_warm(model, split, on="val", to_read=to_read, n_users=1000, seed=42)
    records.append({"model": name, "goal": "warm_val", "history_size": "train", **warm})
    print(f"{name}: warm 평가 {perf_counter() - start:.2f}초")
    start = perf_counter()
    cold = evaluate_cold(model, split, on="tune", to_read=to_read)
    for size, metrics in cold.iterrows():
        records.append({"model": name, "goal": "cold_tune", "history_size": str(size), **metrics.to_dict()})
    print(f"{name}: cold 평가 {perf_counter() - start:.2f}초")
scores = pd.DataFrame(records)
print(scores.to_string(index=False))
scores.to_csv(OUT / "B3_validation_scores.csv", index=False)


popularity: warm 평가 0.25초
popularity: cold 평가 1.86초
cb: warm 평가 0.99초
cb: cold 평가 6.98초
     model      goal history_size  precision@10  recall@10  ndcg@10  users  to_read_hit@10  to_read_users
popularity  warm_val        train      0.036446   0.032146 0.041433  996.0        0.027956          905.0
popularity cold_tune            1      0.207240   0.030281 0.221323 2500.0        0.020148         2298.0
popularity cold_tune            3      0.207240   0.030281 0.221323 2500.0        0.020148         2298.0
popularity cold_tune            5      0.207240   0.030281 0.221323 2500.0        0.020148         2298.0
popularity cold_tune           10      0.207240   0.030281 0.221323 2500.0        0.020148         2298.0
        cb  warm_val        train      0.041365   0.039683 0.048862  996.0        0.042320          905.0
        cb cold_tune            1      0.125360   0.019320 0.139881 2500.0        0.022367         2298.0
        cb cold_tune            3      0.138840   0.021037 0.153

## 3. 인기순 대체 비율

일부 독자는 이력에 4·5점 책이 없어 CB 대신 인기순으로 추천받습니다.
대체 비율을 기록해 첫 평점이 낮은 경우의 한계를 숨기지 않습니다.


In [4]:
users = np.random.default_rng(42).choice(np.sort(split.val["user_id"].unique()), size=1000, replace=False)
histories = [("warm_val", "train", split.train[split.train["user_id"].isin(users)])]
position = split.cold_tune.groupby("user_id").cumcount()
for size in (1, 3, 5, 10):
    histories.append(("cold_tune", str(size), split.cold_tune[position < size]))
diagnostics = []
for goal, size, history in histories:
    summary = models["cb"].profile_summary(history)
    diagnostics.append({"goal": goal, "history_size": size, "users": len(summary),
                        "fallback_users": int(summary["fallback"].sum()),
                        "fallback_rate": float(summary["fallback"].mean())})
diagnostics = pd.DataFrame(diagnostics)
print(diagnostics.to_string(index=False))
diagnostics.to_csv(OUT / "B3_profile_summary.csv", index=False)


     goal history_size  users  fallback_users  fallback_rate
 warm_val        train   1000               0         0.0000
cold_tune            1   2500             616         0.2464
cold_tune            3   2500             158         0.0632
cold_tune            5   2500              67         0.0268
cold_tune           10   2500              15         0.0060


## 4. 학습에 없는 독자의 추천 사례

학습에 없는 독자에게 4점 책 한 권을 이력으로 전달합니다. 그 책을 제외한 추천 10권과 점수 범위·중복을 확인합니다.


In [5]:
new_user = int(split.cold_tune["user_id"].iloc[0])
assert new_user not in set(split.train["user_id"])
history = pd.DataFrame({"user_id": [new_user], "book_id": [1], "rating": [4]})
recommendations = models["cb"].recommend(history, k=10)[new_user]
values = models["cb"].score(history)
assert 1 not in recommendations and len(set(recommendations)) == 10
assert np.isfinite(values.to_numpy()).all()
assert ((values.to_numpy() >= 0) & (values.to_numpy() <= 1)).all()
titles = load_books().set_index("book_id")["title"]
example = pd.DataFrame({"book_id": recommendations})
example["title"] = example["book_id"].map(titles)
example["score"] = values.loc[new_user, recommendations].to_numpy()
print(example.to_string(index=False))


 book_id                                                    title    score
      17                     Catching Fire (The Hunger Games, #2) 0.934240
      20                        Mockingjay (The Hunger Games, #3) 0.872879
     507 The Hunger Games Trilogy Boxset (The Hunger Games, #1-3) 0.829800
      12                                Divergent (Divergent, #1) 0.640206
     327                                      Legend (Legend, #1) 0.557104
      69                                Insurgent (Divergent, #2) 0.553801
      91                        The Maze Runner (Maze Runner, #1) 0.544545
    2301               The Ask and the Answer (Chaos Walking, #2) 0.529215
     187                                      Uglies (Uglies, #1) 0.527827
    1413            Under the Never Sky (Under the Never Sky, #1) 0.521297


## 다음 작업

B4에서는 같은 분할·검증 독자로 특징 가중치와 독자 가중치를 비교합니다.
현재 결과가 인기순보다 낮더라도 기록을 유지합니다. 이 모델은 코사인 점수를 반환하므로 RMSE를 계산하지 않습니다.
hybrid 담당자는 `fit(split.train)` 뒤 `score(history)`를 호출하면 됩니다. 이미 읽은 책 제거는 공통 recommend가 처리합니다.
